Today is 27/11/25. We start this journey. 27 of the month is a good day ... let's hope this will be a good luck sign :)

In [18]:
from gisele.preprocessing import *
from gisele.clustering import * 
from gisele import MILP_models
from importlib import reload
from gisele import postprocessing 

reload(MILP_models) 
reload(postprocessing) 

<module 'gisele.postprocessing' from 'c:\\Users\\corra\\Desktop\\GISEle_v1.0\\gisele\\postprocessing.py'>

In [2]:

case_study = 'Lesotho_1'
country = 'Lesotho' 
crs = 'EPSG:21037'  # Lesotho 'EPSG:21037'; Uganda 'EPSG:21095'


The following step evaluates the existence of the same case study. If you are running a new case study, a new folder structure will be created. Initial information must be load in the folder Database 

Clusters will contain the geopanda dataframe if clusters exists otherwise Clusters will be a boolean = True to show that clustering is needed. 

Note that Clusters, study_area and Substations are saved reprojected in the crs

In [3]:
Clusters, study_area, Substations = create_folder(case_study, country, crs) 


The following step evalautes clustering. Some input information are needed: 
- Open Energy Map: are you using open energy map data?
- household: are you using household data 
- Urbanity: do you have urbanity raster to use Improved DBSCAN? See documentation for more 


In [4]:

if Clusters is True:
    household = True 
    urbanity_flag = True  
    if household: 
        areaLowerBound = 1 # household with a impronta lower than this value will be discarded  

        radius = 30 #Parameters for community clustering 
        dens_filter = 30 #Parameters for community clustering 


        threshold = 0.99 #Electrification Threshold in case you are working with open energy map data 
        
        Cluster = clustering.building_to_cluster_v1(crs,case_study, study_area, country, urbanity_flag, areaLowerBound, radius, dens_filter, threshold)

    else: #this means you are using a raster based population dataset 
        print("Exit  3")
else: 
    print("No need to clustering - We have it already") 
n_clusters = Clusters.shape[0] 

No need to clustering - We have it already


Here we perform MILP optimization. There is an input section to set parameters. 

In [11]:
distance_constraint = False 
voltage_constraint = True 
Abase = 1 
Vmin = 0.9 
mg_option = False 
reliability_option = False 
n_line_type=1 
coe = 10 
voltage = 50# [kV] 15kV - Italy , 11kV Lesotho 


In [12]:
resistance = 0.306
reactance= 0.33
Pmax = 6.69 # MVA, in amperes it is 350A
line_cost = 25000 # not clear if this is for one phase or for all 3
line1 = {"resistance":resistance,"reactance":reactance,"Pmax":Pmax,"line_cost":line_cost}

In [19]:

MILP_models.MILP_base(case_study,n_clusters,coe,voltage,line1)

Instance is constructed: True
Starting optimization process
Read LP format model from file C:\Users\corra\AppData\Local\Temp\tmpp102wxki.pyomo.lp
Reading time = 0.03 seconds
Obj: 1592 rows, 734 columns, 3947 nonzeros
Set parameter MIPGap to value 0.14
Set parameter Presolve to value 2
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i9-14900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  0.14
Presolve  2

Optimize a model with 1592 rows, 734 columns and 3947 nonzeros (Min)
Model fingerprint: 0xa2e04c5a
Model has 189 linear objective coefficients
Variable types: 545 continuous, 189 integer (189 binary)
Coefficient statistics:
  Matrix range     [4e-06, 1e+01]
  Objective range  [9e+03, 8e+06]
  Bounds range     [1e+00, 1e+00]
  RHS range        [8e-04, 7e+01]
Presolve removed 651 rows and 151 columns
Presolve time: 0.01s
Presol

Postprocessing section 


In [20]:
postprocessing.process(case_study,crs,mg_option,reliability_option)
postprocessing.create_final_output(case_study)
if mg_option == True:
    postprocessing.analyze(case_study,coe,mg_option,n_line_type)

Only on-grid


c:\Users\corra\Desktop\GISEle_v1.0\gisele\postprocessing.py:157: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  Length.append(round(abs(float(distance)),3))
c:\Users\corra\Desktop\GISEle_v1.0\gisele\postprocessing.py:158: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  Cost.append(float(cost))
c:\Users\corra\Desktop\GISEle_v1.0\gisele\postprocessing.py:193: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  x.append(float(Nodes.loc[(Nodes['ID']==id),'X']))
c:\Users\corra\Desktop\GISEle_v1.0\gisele\postprocessing.py:194: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  y.append(float(Nodes.loc[(Nodes['ID']==i

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\corra\\Desktop\\GISEle_v1.0\\Case studies\\Lesotho_1\\Output\\LV_resume.csv'

In [ ]:
"C:\Users\corra\Desktop\GISEle_v1.0\Case studies\Lesotho_1\Intermediate\Optimization\MILP_output\connections_output.csv"